In [0]:

# TSI - Validación automatizada de datos
# Operaciones de solo lectura

from pyspark.sql import functions as F

TABLA = "workspace.tsi.international_modeling"

df = spark.table(TABLA)

resumen = (
    df.groupBy("fuente")
    .agg(
        F.count("*").alias("observaciones"),
        F.min("timestamp").alias("primera_fecha"),
        F.max("timestamp").alias("ultima_fecha")
    )
    .orderBy("fuente")
)

display(resumen)


In [0]:

# Comprobación de fuentes esperadas

fuentes_esperadas = {
    "istanbul_traffic_index",
    "madrid_traffic_points",
    "metr_la",
    "pems_bay",
    "sao_paulo_cet"
}

fuentes_encontradas = {
    fila["fuente"]
    for fila in resumen.select("fuente").collect()
}

faltantes = fuentes_esperadas - fuentes_encontradas

if faltantes:
    raise ValueError(
        f"Faltan fuentes en Databricks: {sorted(faltantes)}"
    )

print("VALIDACIÓN EXITOSA")
print("Las cinco fuentes internacionales están disponibles.")
